In [0]:
import pandas as pd
import itertools
import random
from collections import defaultdict

In [0]:
spark_df = spark.sql("select * from default.retail_sales_dataset")
df = spark_df.toPandas()

df.head()

# Nested & List Comprehensions

In [0]:
categories = df["Product Category"].unique().tolist()
genders    = df["Gender"].unique().tolist()

In [0]:
categories

In [0]:
print("\nRevenue Matrix — Category × Gender\n")
for category in categories:
    for gender in genders:
        mask = (df["Product Category"] == category) & (df["Gender"] == gender)
        revenue = df.loc[mask, "Total Amount"].sum()
        print(f"  {category:<15} | {gender:<8} | R{revenue:>10}")


In [0]:
age_bands

In [0]:
print("\nTransaction Count — Age Band × Category\n")

age_bands = [("18-29", 18, 29), ("30-44", 30, 44), ("45-64", 45, 64)]

for band_label, low, high in age_bands:
    for category in categories:
        mask = (df["Age"] >= low) & (df["Age"] <= high) & (df["Product Category"] == category)
        count = mask.sum()
        print(f"  {band_label:<8} | {category:<15} | {count:>4} transactions")




In [0]:
for i, x in itertools.combinations(categories, 2):
    print(i, x)

In [0]:
import itertools

In [0]:
print("\nAvg Spend Comparison — All Category Pairs\n")
for cat_a, cat_b in itertools.combinations(categories, 2):
    avg_a = df.loc[df["Product Category"] == cat_a, "Total Amount"].mean()
    avg_b = df.loc[df["Product Category"] == cat_b, "Total Amount"].mean()
    
    higher = cat_a if avg_a > avg_b else cat_b
    
    print(f"  {cat_a} (R{avg_a:.0f}) vs {cat_b} (R{avg_b:.0f})  →  {higher} wins")

In [0]:
print([amt for amt in df["Total Amount"] if amt > 500])

In [0]:
for amt in df["Total Amount"]:
     if amt > 500:
         print(amt)

In [0]:
print("\nTransactions over R500\n")
high_value = [amt for amt in df["Total Amount"] if amt > 500]

print(f"  Count : {len(high_value)}")
print(f"  Total : R{sum(high_value):,.0f}")
print(f"  Sample: {high_value[:10]}")

In [0]:
pd.Series(spend_labels).value_counts()

In [0]:
print("\n Spend-Level Labels per Transaction (first 10)\n")
def classify_spend(amount):
    if amount < 100:   return "Low"
    elif amount < 500: return "Medium"
    else:              return "High"

spend_labels = [classify_spend(amt) for amt in df["Total Amount"]]

df["Spend Level"] = spend_labels

print(f"  First 10 labels : {spend_labels[:10]}")
print(f"  Distribution    : {pd.Series(spend_labels).value_counts().to_dict()}")

In [0]:
print("\nRevenue Matrix — Category × Gender\n")
for category in categories:
    for gender in genders:
        mask = (df["Product Category"] == category) & (df["Gender"] == gender)
        revenue = df.loc[mask, "Total Amount"].sum()
        print(f"  {category:<15} | {gender:<8} | R{revenue:>10}")

In [0]:
print("\nNested List Comprehension — Revenue Grid\n")
revenue_grid = [
    [df.loc[(df["Product Category"] == cat) & (df["Gender"] == gen),
            "Total Amount"].sum()
     for gen in genders]
    for cat in categories
]


header = f"  {'Category':<15} " + "  ".join(f"{g:>10}" for g in genders)
print(header)


for cat, row in zip(categories, revenue_grid):
    print(f"  {cat:<15} " + "  ".join(f"R{v:>9}" for v in row))

In [0]:
df.head()

# THings are getting deep


- The raw dataset has 1 transaction per Customer ID.
- To demonstrate basket analysis, we simulate "households" by randomly
- grouping 1-3 transactions together.  This is transparent to students —
- it's a great teachable moment about when you need to engineer your keys.


[Context] Each row in our dataset is one transaction.
  For basket analysis we need to group transactions by "shopper visit."
  Because Customer IDs are unique here, we'll simulate households
  by pooling transactions that share the same Date and Gender —
  a realistic proxy for co-purchases on the same shopping trip.

In [0]:
# Build baskets (grouped by Date × Gender as proxy for visit)
print("Build Customer Baskets (Date × Gender grouping)\n")

baskets_raw = df.groupby(["Date", "Gender"])["Product Category"].apply(set)
baskets = [b for b in baskets_raw if len(b) > 1]          # only multi-item


single_item_baskets = [b for b in baskets_raw if len(b) == 1]


print(f"  Total visit groups      : {len(baskets_raw)}")
print(f"  Multi-category baskets  : {len(baskets)}")
print(f"  Single-category baskets : {len(single_item_baskets)}")
print(f"\n  Sample multi-item baskets:")
for b in baskets[:6]:
    print(f"    {sorted(b)}")

In [0]:
all_baskets = list(baskets_raw)   # include singles for support calculation
n_baskets   = len(all_baskets)

In [0]:
# Item Support
print("\nItem Support (% of baskets containing each category)\n")
support = {}
for cat in categories:
    count = sum(1 for b in all_baskets if cat in b)
    support[cat] = count / n_baskets
    print(f"  {cat:<15}: {support[cat]*100:.1f}% ({count}/{n_baskets} baskets)")

In [0]:
print("\nPair Co-occurrence — Nested Loop Over All Pairs\n")

pair_counts = {}

for cat_a, cat_b in itertools.combinations(categories, 2):
    co_count = sum(1 for b in all_baskets if cat_a in b and cat_b in b)
    
    pair_counts[(cat_a, cat_b)] = co_count
    
    print(f"  {cat_a} + {cat_b:<20}: {co_count} co-occurring baskets")

In [0]:
pair_counts


In [0]:
print("\n Confidence  P(B | A) — 'Customers who bought A also bought B'\n")
for (cat_a, cat_b), co_count in pair_counts.items():
    count_a = sum(1 for b in all_baskets if cat_a in b)
    count_b = sum(1 for b in all_baskets if cat_b in b)
    
    conf_ab = co_count / count_a if count_a else 0
    conf_ba = co_count / count_b if count_b else 0
    
    
    print(f"  P({cat_b:<15} | {cat_a:<15}) = {conf_ab*100:5.1f}%")
    print(f"  P({cat_a:<15} | {cat_b:<15}) = {conf_ba*100:5.1f}%")
    print()


In [0]:

# Lift  (the "is this more than coincidence?" metric)
print("  Lift  =  Confidence(A→B) / Support(B)\n")
print("  Lift > 1  →  positive association (co-purchase more likely than chance)")
print("  Lift = 1  →  independent")
print("  Lift < 1  →  negative association\n")

print(f"  {'Rule':<35} {'Support':>9} {'Confidence':>11} {'Lift':>8}")
print("  " + "-" * 68)

rules = []
for (cat_a, cat_b), co_count in pair_counts.items():
    count_a = sum(1 for b in all_baskets if cat_a in b)
    count_b = sum(1 for b in all_baskets if cat_b in b)
    sup_ab  = co_count / n_baskets
    conf_ab = co_count / count_a if count_a else 0
    conf_ba = co_count / count_b if count_b else 0
    
    
    lift_ab = conf_ab / support[cat_b] if support[cat_b] else 0
    lift_ba = conf_ba / support[cat_a] if support[cat_a] else 0
    rules += [
        (f"{cat_a} → {cat_b}", sup_ab, conf_ab, lift_ab),
        (f"{cat_b} → {cat_a}", sup_ab, conf_ba, lift_ba),
    ]

rules.sort(key=lambda x: x[3], reverse=True)
for rule, sup, conf, lift in rules:
    print(f"  {rule:<35} {sup*100:>8.1f}% {conf*100:>10.1f}% {lift:>8.3f}")



──────────────────
CLASS DISCUSSION QUESTIONS
──────────
1. Which product pair has the strongest association?
   What marketing action would you recommend?

2. A lift of 1.0 means independence.  What does a
   lift < 1 mean from a business perspective?

3. How would you use these rules in a recommender
   system or store layout strategy?

4. What limitation does our basket-building method
   (Date × Gender grouping) introduce?  How would
   you improve it with richer data?